
# Federated Learning Training with Flower and TensorFlow

This notebook implements federated training on the Fashion-MNIST dataset
using Flower (`flwr`) and TensorFlow. Each client trains a simple CNN
a locally and the Flower server aggregates weights using FedAvg.

**Usage**:
- Run the server cell to start the aggregator.
- Launch client cells (in separate processes or terminals) with different `cid` values.

Requirements:
```bash
pip install flwr tensorflow
```


In [1]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
import flwr as fl

# Hyperparameters
NUM_CLIENTS = 5
ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
SEED = 42
np.random.seed(SEED)


2025-07-21 09:47:15.198263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-21 09:47:17,455	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:

def load_partition(cid: int):
    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0
    x_train = x_train.reshape(-1, 28, 28, 1)
    x_test = x_test.reshape(-1, 28, 28, 1)

    size_per_client = len(x_train) // NUM_CLIENTS
    start, end = cid * size_per_client, (cid + 1) * size_per_client
    return (x_train[start:end], y_train[start:end]), (x_test, y_test)


In [3]:

def build_model():
    model = keras.Sequential([
        keras.layers.Conv2D(8, 3, strides=2, activation="relu", input_shape=(28, 28, 1)),
        keras.layers.Flatten(),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


In [4]:

class FashionClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.cid = cid
        (self.x_train, self.y_train), (self.x_test, self.y_test) = load_partition(cid)
        self.model = build_model()

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)
        self.model.fit(self.x_train, self.y_train,
                       epochs=LOCAL_EPOCHS,
                       batch_size=BATCH_SIZE,
                       verbose=0)
        return self.model.get_weights(), len(self.x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        loss, acc = self.model.evaluate(self.x_test, self.y_test, verbose=0)
        return loss, len(self.x_test), {"accuracy": acc}


In [5]:

def start_server():
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
    )
    fl.server.start_server(server_address="0.0.0.0:8080",
                           config=fl.server.ServerConfig(num_rounds=ROUNDS),
                           strategy=strategy)


def start_client(cid):
    fl.client.start_numpy_client(server_address="0.0.0.0:8080",
                                 client=FashionClient(cid))

# To start, uncomment the desired cell below:
#start_server()


In [6]:
import flwr as fl
from flwr.common import Context
from flwr.client import ClientApp
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation

# 1) Client factory: ora prende un Context, non il cid diretto
def client_fn(context: Context) -> fl.client.NumPyClient:
    # context.node_id è l'indice (int) del client
    return FashionClient(context.node_id)

# 2) Server factory: crea la strategia e la config
def server_fn() -> ServerAppComponents:
    strategy = FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )
    config = ServerConfig(num_rounds=ROUNDS)
    return ServerAppComponents(strategy=strategy, config=config)

# 3) Istanze di ServerApp e ClientApp
server_app = ServerApp(server_fn=server_fn)
client_app = ClientApp(client_fn=client_fn)

# 4) Avvia la simulazione in‐process e termina dopo ROUNDS round
history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,      # <-- il numero di client da simulare
    # backend_name="ray",            # (opzionale)
    # backend_config=None,           # (opzionale)
    # enable_tf_gpu_growth=False,    # (opzionale)
    # verbose_logging=False,         # (opzionale)
)

# (Opzionale) Stampare metriche
print("Fit metrics:", history.metrics_fit)
print("Evaluate metrics:", history.metrics_evaluate)


ERROR :     ServerApp thread raised an exception: server_fn() takes 0 positional arguments but 1 was given
ERROR :     Traceback (most recent call last):
  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/jupyter-workflow/examples/tensorflow/work/TestTrainingModelOld/fedenv/lib/python3.11/site-packages/flwr/simulation/run_simulation.py", line 268, in server_th_with_start_checks
    updated_context = _run(
                      ^^^^^
  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/jupyter-workflow/examples/tensorflow/work/TestTrainingModelOld/fedenv/lib/python3.11/site-packages/flwr/server/run_serverapp.py", line 62, in run
    server_app(grid=grid, context=context)
  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/jupyter-workflow/examples/tensorflow/work/TestTrainingModelOld/fedenv/lib/python3.11/site-packages/flwr/server/server_app.py", line 161, in __call__
    components = self._server_fn(context)
                 ^^^^^^^^^^^^^^^^^^^^^^^^
TypeError

RuntimeError: Exception in ServerApp thread